# Лабораторна робота 4 — оцінювання якості даних
## Melbourne Housing FULL

**Набір даних:** `Melbourne_housing_FULL.csv` (Melbourne housing market).

**Мета:** оцінити придатність даних для описового аналізу ринку житла та побудови моделей ціни. Профіль, метрики й перевірки обчислюються з фактичного CSV під час запуску; наведені нижче пороги є робочими вимогами аналітичного сценарію, а не зовнішнім стандартом.

> Запустіть ноутбук послідовно зверху вниз. Коментарі в коді написані англійською. Рядки, що містять `NaN`, не видаляються з початкового набору; очищена копія використовується лише для аналізу заглушок.

## 1. Вимоги до якості

### Призначення та ризики
Дані описують оголошення/продажі житла у Мельбурні. Їх може використовувати аналітик ринку для порівняння районів, а дослідник — для прогнозування ціни. Некоректна ціна або географія спотворює висновки й прогноз; пропущена площа чи рік будівництва обмежує вибірку моделей; повторні записи завищують вагу окремих об'єктів. CSV не містить надійного ідентифікатора транзакції чи часу завантаження, тому унікальність та актуальність оцінюються за проксі й мають обмеження.

| Вимір | Метрика | Робочий поріг | Обґрунтування |
|---|---|---:|---|
| Повнота | Заповнені значення / усі рядки для поля | Price ≥ 80%; Rooms, Suburb ≥ 99%; інші поля — звітний показник | Ціна є цільовою змінною, а базові ознаки мають бути майже завжди доступні. Поріг Price дозволяє аналізувати більшу частину набору без припущення, що пропуск випадковий. |
| Унікальність | 1 − частка повторів кандидатного ключа `Suburb + Address + Date + Method` | ≥ 98% | Один запис на адресу, дату й метод продажу є практичним кандидатом на подію; повтори слід перевіряти, а не автоматично видаляти. |
| Валідність | Частка непорожніх значень, що проходять доменні перевірки | ≥ 99% для базових доменів (ціна, кімнати, тип) | Неможливі або невідомі значення не повинні непомітно потрапляти в агрегати чи моделі. |
| Узгодженість | Частка записів без порушень між пов'язаними полями | ≥ 99% | Суперечливі характеристики одного об'єкта знижують довіру до запису. |
| Актуальність | Частка дат, не старших за 365 днів від останньої дати набору | ≥ 80% для сценарію моніторингу | CSV містить історичні спостереження. Вік рахується від максимальної дати у файлі, а не від сьогодні, бо дата вивантаження невідома. |
| Репрезентативність зрізів | Розмір групи та відхилення повноти Price від загальної | Кожен звітний регіон має ≥ 100 рядків; відхилення повноти ≤ 10 в.п. | Великі локальні дефекти важливо бачити окремо, навіть якщо загальна метрика прийнятна. |

**Припущення:** порожні CSV-поля означають невідоме/не надане значення, а не нуль; `Date` — дата події у форматі день/місяць/рік; назви типів нерухомості очікуються `h`, `u`, `t`; значення `Price` мають бути додатними AUD; `Suburb + Address + Date + Method` — лише кандидатний ключ, що може мати легітимні повтори; дата завантаження відсутня. Заглушки перевіряються за контекстом стовпця, щоб не перетворити коректні нулі координат чи кількості автомобілів на пропуски.

In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

# Use the provided dataset path when available, with a portable local fallback.
DATA_PATH = Path(os.environ.get("MELBOURNE_DATA", Path.cwd() / "Melbourne_housing_FULL.csv"))
if not DATA_PATH.exists():
    DATA_PATH = Path(r"C:\Users\Kate\Desktop\Унік\мага\sem_1\ляшкевич\Melbourne_housing_FULL.csv")
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset not found: {DATA_PATH}. Set MELBOURNE_DATA to its location.")

df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(df):,} rows and {df.shape[1]} columns from {DATA_PATH}")
display(df.head())
display(pd.DataFrame({"dtype": df.dtypes.astype(str), "non_null": df.notna().sum(), "missing": df.isna().sum()}))

## 2. Профіль набору даних і пропуски
Функція `profile` показує тип, частку пропусків, унікальність, мінімум/максимум і п'ять найчастіших значень для кожного поля. Для текстових і датоподібних полів мінімум/максимум є лексикографічними або часовими орієнтирами, а не числовою оцінкою.

In [ ]:
def profile(data):
    # Summarize completeness, cardinality, range, and common values per column.
    rows = []
    for column in data.columns:
        series = data[column]
        non_null = series.dropna()
        try:
            minimum, maximum = non_null.min(), non_null.max()
        except (TypeError, ValueError):
            minimum, maximum = None, None
        top = series.value_counts(dropna=True).head(5).to_dict()
        rows.append({
            "column": column, "dtype": str(series.dtype),
            "missing_count": int(series.isna().sum()),
            "missing_share": float(series.isna().mean()),
            "unique_count": int(series.nunique(dropna=True)),
            "unique_share": float(series.nunique(dropna=True) / max(series.notna().sum(), 1)),
            "min": minimum, "max": maximum, "top_5": top,
        })
    return pd.DataFrame(rows).set_index("column")

raw_profile = profile(df)
display(raw_profile)
print(f"Exact duplicate rows: {df.duplicated().sum():,} ({df.duplicated().mean():.2%})")

### Пошук контекстних заглушок
Заглушки треба визначати за змістом поля. Наприклад, `0` може бути коректним для `Car`, а `1900` не є надійно підтвердженим роком будівництва. Код рахує кандидати до перетворення; числові -1/9999 та рік 1900 замінюються лише у визначених полях. Текстові `N/A` сприймаються pandas як пропуски під час читання CSV.

In [ ]:
# Count placeholder candidates before cleaning, scoped to semantically relevant columns.
placeholder_counts = {}
for column in df.columns:
    series = df[column]
    count = int(series.astype(str).str.strip().str.upper().isin({"N/A", "NA", "NULL", "UNKNOWN"}).sum())
    if count:
        placeholder_counts[column] = count
for column in ["Price", "Rooms", "Bedroom2", "Bathroom", "Car", "Landsize", "BuildingArea", "YearBuilt", "Distance", "Postcode", "Propertycount"]:
    if column in df:
        numeric = pd.to_numeric(df[column], errors="coerce")
        count = int(numeric.isin([-1, 9999]).sum())
        if count:
            placeholder_counts[column] = placeholder_counts.get(column, 0) + count
        if column == "YearBuilt":
            count_1900 = int(numeric.eq(1900).sum())
            if count_1900:
                placeholder_counts[column] = placeholder_counts.get(column, 0) + count_1900
display(pd.Series(placeholder_counts, name="placeholder_candidates").to_frame())


### Normalize placeholders

Convert only context-specific numeric placeholders to `NaN`, then profile the cleaned data again.

In [ ]:
clean = df.copy()
for column in ["Price", "Rooms", "Bedroom2", "Bathroom", "Car", "Landsize", "BuildingArea", "YearBuilt", "Distance", "Postcode", "Propertycount"]:
    if column in clean:
        clean[column] = pd.to_numeric(clean[column], errors="coerce")
        clean.loc[clean[column].isin([-1, 9999]), column] = np.nan
if "YearBuilt" in clean:
    clean.loc[clean["YearBuilt"].eq(1900), "YearBuilt"] = np.nan

print("Missing values before/after contextual placeholder handling:")
display(pd.DataFrame({"before": df.isna().sum(), "after": clean.isna().sum(), "newly_missing": clean.isna().sum() - df.isna().sum()}))
clean_profile = profile(clean)
display(clean_profile[["dtype", "missing_count", "missing_share", "unique_count", "min", "max"]])

In [ ]:
# Plot a readable missingness map for a deterministic sample of rows.
sample_size = min(1500, len(clean))
missing_sample = clean.sample(sample_size, random_state=42).isna()
plt.figure(figsize=(15, 6))
sns.heatmap(missing_sample, cbar=False, yticklabels=False, cmap="magma")
plt.title(f"Missing-value map (sample of {sample_size:,} rows)")
plt.xlabel("Columns")
plt.tight_layout()
plt.show()

missing_by_column = clean.isna().mean().sort_values(ascending=False)
display(missing_by_column.rename("missing_share").to_frame().head(10))
print("Rows with at least one missing field:", f"{clean.isna().any(axis=1).mean():.2%}")

## 3. Метрики якості за стовпцями та зрізами
Метрики повноти, кандидатної унікальності, доменної валідності, узгодженості та актуальності зводяться в єдину таблицю. Показник актуальності тут — частка дат не старших за один рік від найновішого спостереження; він не доводить актуальність щодо реального часу. Для зрізів використано регіон і рік події, оскільки джерело та дата завантаження у файлі відсутні.

In [ ]:
# Parse dates explicitly because the source uses day/month/year strings.
if "Date" in clean:
    clean["Date_parsed"] = pd.to_datetime(clean["Date"], dayfirst=True, errors="coerce")

def share(mask, denominator=None):
    # Return a safe fraction for boolean masks.
    if denominator is None:
        denominator = len(mask)
    return float(mask.sum() / denominator) if denominator else np.nan

n = len(clean)
metrics = []
def add_metric(dimension, field, value, threshold, note=""):
    metrics.append({"dimension": dimension, "field": field, "value": value, "threshold": threshold, "status": "PASS" if pd.notna(value) and value >= threshold else "REVIEW", "note": note})

for column, threshold in [("Price", 0.80), ("Rooms", 0.99), ("Suburb", 0.99), ("BuildingArea", 0.50), ("YearBuilt", 0.50)]:
    if column in clean:
        add_metric("completeness", column, float(clean[column].notna().mean()), threshold)

key_columns = [c for c in ["Suburb", "Address", "Date", "Method"] if c in clean]
complete_key = clean[key_columns].notna().all(axis=1)
key_unique_share = float(1 - clean.loc[complete_key].duplicated(key_columns).mean()) if complete_key.any() else np.nan
add_metric("uniqueness", " + ".join(key_columns), key_unique_share, 0.98, "Candidate event key; duplicates need review")


### Validity, consistency, and timeliness

Check value domains, relationships between fields, and date age within the dataset.

In [ ]:
if "Price" in clean:
    valid_price = clean["Price"].gt(0) & clean["Price"].notna()
    add_metric("validity", "Price > 0", share(valid_price, clean["Price"].notna().sum()), 0.99)
if "Rooms" in clean:
    valid_rooms = clean["Rooms"].between(1, 10)
    add_metric("validity", "Rooms in [1, 10]", share(valid_rooms, clean["Rooms"].notna().sum()), 0.99)
if "Type" in clean:
    valid_type = clean["Type"].isin(["h", "u", "t"])
    add_metric("validity", "Type in {h,u,t}", share(valid_type, clean["Type"].notna().sum()), 0.99)
if {"Bedroom2", "Rooms"}.issubset(clean.columns):
    checked = clean[["Bedroom2", "Rooms"]].notna().all(axis=1)
    consistent = clean["Bedroom2"].le(clean["Rooms"])
    add_metric("consistency", "Bedroom2 <= Rooms", share(consistent[checked], int(checked.sum())), 0.99, "Heuristic; source semantics may differ")
if {"Lattitude", "Longtitude"}.issubset(clean.columns):
    checked = clean[["Lattitude", "Longtitude"]].notna().all(axis=1)
    consistent = clean["Lattitude"].between(-39, -37) & clean["Longtitude"].between(144, 146)
    add_metric("consistency", "Melbourne coordinate bounds", share(consistent[checked], int(checked.sum())), 0.99)
if "Date_parsed" in clean and clean["Date_parsed"].notna().any():
    newest = clean["Date_parsed"].max()
    recent = clean["Date_parsed"].ge(newest - pd.Timedelta(days=365))
    add_metric("timeliness", "Date within 365 days of dataset maximum", share(recent[clean["Date_parsed"].notna()], int(clean["Date_parsed"].notna().sum())), 0.80, f"Reference date: {newest.date()}")


In [ ]:
quality_metrics = pd.DataFrame(metrics)
display(quality_metrics)

In [ ]:
# Compare price completeness and validity across region and event-year slices.
if "Regionname" in clean:
    region_quality = clean.groupby("Regionname", dropna=False).agg(
        rows=("Suburb", "size"),
        price_completeness=("Price", lambda s: s.notna().mean()),
        median_price=("Price", "median"),
    ).sort_values("price_completeness")
    display(region_quality)
    eligible = region_quality[region_quality["rows"] >= 100]
    print("Lowest completeness region with at least 100 rows:", eligible.index[0] if len(eligible) else "No eligible region")

if "Date_parsed" in clean and "Price" in clean:
    clean["event_year"] = clean["Date_parsed"].dt.year
    year_quality = clean.groupby("event_year", dropna=False).agg(
        rows=("Price", "size"),
        price_completeness=("Price", lambda s: s.notna().mean()),
    ).sort_index()
    display(year_quality)


### Slice comparison plots

The plots make local differences visible even when the overall missing-value rate looks acceptable.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
if "Regionname" in clean:
    region_quality["price_completeness"].plot(kind="bar", ax=axes[0], color="steelblue")
    axes[0].axhline(0.80, color="red", linestyle="--", label="threshold = 0.80")
    axes[0].set_title("Price completeness by region")
    axes[0].set_ylabel("Completeness")
    axes[0].legend()
if "Date_parsed" in clean and "Price" in clean:
    year_quality["price_completeness"].plot(ax=axes[1], marker="o", color="darkgreen")
    axes[1].axhline(0.80, color="red", linestyle="--", label="threshold = 0.80")
    axes[1].set_title("Price completeness by event year")
    axes[1].set_ylabel("Completeness")
    axes[1].legend()
plt.tight_layout()
plt.show()


In [ ]:
# Report how far the worst sufficiently large region differs from the overall rate.
overall_price_completeness = clean["Price"].notna().mean()
if "Regionname" in clean and len(eligible):
    worst_region = eligible.index[0]
    gap_pp = (overall_price_completeness - eligible.loc[worst_region, "price_completeness"]) * 100
    print(f"Overall Price completeness: {overall_price_completeness:.2%}; {worst_region}: {eligible.loc[worst_region, 'price_completeness']:.2%} (gap {gap_pp:.1f} percentage points).")
    print("A global average can hide this local defect because better-performing regions offset missing prices in the weaker region.")

## 4. Автоматизовані перевірки
Нижче реалізовано перевірки на рівні полів, записів і набору. `critical` означає блокування сценарію, що вимагає надійної цільової ціни/ідентичності подій; `warning` означає, що аналіз можна продовжити після перегляду звіту. Для кожного правила звіт містить кількість, частку та приклади рядків.

In [ ]:
def build_check_report(data):
    # Collect row-level violations and summarize each rule with examples.
    checks = []
    def add(name, mask, severity, description):
        mask = pd.Series(mask, index=data.index).fillna(False).astype(bool)
        checks.append((name, mask, severity, description))
    n_rows = len(data)
    if "Price" in data:
        add("Price missing", data["Price"].isna(), "critical", "Target price is missing")
        add("Price non-positive", data["Price"].notna() & data["Price"].le(0), "critical", "Price must be positive")
        add("Price extreme", data["Price"].notna() & ~data["Price"].between(10_000, 20_000_000), "warning", "Review price outside broad AUD range")
    if "Rooms" in data:
        add("Rooms integer domain", data["Rooms"].notna() & (~data["Rooms"].mod(1).eq(0) | ~data["Rooms"].between(1, 10)), "warning", "Rooms must be an integer from 1 to 10")
    if "Type" in data:
        add("Type domain", data["Type"].notna() & ~data["Type"].isin(["h", "u", "t"]), "warning", "Unexpected property type")
    if "Suburb" in data:
        add("Suburb present", data["Suburb"].isna() | data["Suburb"].astype(str).str.strip().eq(""), "critical", "Suburb is required for geographic analysis")
    if "Date_parsed" in data:
        add("Date parseable", data["Date_parsed"].isna(), "warning", "Date is missing or cannot be parsed")
    if "BuildingArea" in data:
        add("BuildingArea positive", data["BuildingArea"].notna() & data["BuildingArea"].le(0), "warning", "Building area must be positive when supplied")
    if {"Bedroom2", "Rooms"}.issubset(data.columns):
        checked = data[["Bedroom2", "Rooms"]].notna().all(axis=1)
        add("Bedrooms consistent with rooms", checked & data["Bedroom2"].gt(data["Rooms"]), "warning", "Bedroom2 exceeds Rooms")
    if {"Lattitude", "Longtitude"}.issubset(data.columns):
        both = data[["Lattitude", "Longtitude"]].notna().all(axis=1)
        inside = data["Lattitude"].between(-39, -37) & data["Longtitude"].between(144, 146)
        add("Coordinates in Melbourne bounds", both & ~inside, "warning", "Coordinates fall outside broad Melbourne bounds")
    candidate_key = [c for c in ["Suburb", "Address", "Date", "Method"] if c in data]
    if candidate_key:
        key_complete = data[candidate_key].notna().all(axis=1)
        duplicated_key = data.loc[key_complete].duplicated(candidate_key, keep=False).reindex(data.index, fill_value=False)
        add("Candidate event key duplicate", duplicated_key, "warning", "Candidate key repeats; inspect before deduplication")
    add("Dataset minimum size", pd.Series(n_rows < 5000, index=data.index), "critical", "Dataset must contain at least 5,000 records")
    rows = []
    for name, violations, severity, description in checks:
        positions = np.flatnonzero(violations.to_numpy())[:3]
        examples = data.iloc[positions].head(3).to_dict(orient="records")
        count = int(violations.sum())
        rows.append({"rule": name, "level": "dataset" if name == "Dataset minimum size" else ("record" if "consistent" in name.lower() or "duplicate" in name.lower() else "field"),
                     "severity": severity, "violations": count, "share": count / max(n_rows, 1),
                     "description": description, "examples": examples})
    return pd.DataFrame(rows)

check_report = build_check_report(clean)
display(check_report.drop(columns="examples"))
display(check_report[["rule", "examples"]])
print("Critical rules with violations:", int(((check_report.severity == "critical") & (check_report.violations > 0)).sum()))
print("Pipeline policy: stop on any critical violation; warn and continue on warning violations after recording examples.")

## 5. Протокол спостережень і матриця якості
Контрольний список покриває розмір, повноту, повтори, домени, зв'язки полів, дати й регіональні зрізи. Таблиці нижче формуються з поточного запуску, тому числові результати не вигадуються наперед. Релевантність дефекту залежить від сценарію: відсутня ціна критична для регресії ціни, але може бути прийнятною для опису кількості оголошень.

In [ ]:
# Build an observation log from the checks and a quality matrix from measured metrics.
observation_log = check_report.loc[check_report["violations"] > 0, ["rule", "severity", "violations", "share", "description", "examples"]].copy()
observation_log["relevance"] = observation_log["severity"].map({"critical": "critical", "warning": "substantial"})
display(observation_log)

quality_matrix = quality_metrics[["dimension", "field", "threshold", "value", "status", "note"]].copy()
quality_matrix["value"] = quality_matrix["value"].map(lambda value: f"{value:.2%}" if pd.notna(value) else "not available")
quality_matrix["threshold"] = quality_matrix["threshold"].map(lambda value: f"{value:.2%}")
display(quality_matrix)


### Completed validation checklist

In [ ]:
checklist = pd.DataFrame([
    ("Dataset size", "Minimum 5,000 rows rule", len(clean), "checked"),
    ("Column profile", "profile(df): type, missingness, uniqueness, range, top values", len(clean.columns), "checked"),
    ("Hidden missing values", "Contextual -1/9999/1900 and text token scan", int(sum(placeholder_counts.values())), "checked"),
    ("Missingness map", "Fixed-seed sample heatmap", sample_size, "checked"),
    ("Quality metrics", "Completeness, uniqueness, validity, consistency, timeliness", len(quality_metrics), "checked"),
    ("Slices", "Region and event-year comparisons", int(clean["Regionname"].nunique()) if "Regionname" in clean else 0, "checked"),
    ("Automated rules", "Field, record, and dataset checks", len(check_report), "checked"),
], columns=["item", "method", "sample_or_count", "status"])
display(checklist)


### Overall suitability conclusion

In [ ]:
failed_metrics = quality_metrics[quality_metrics["status"] != "PASS"]
critical_violations = int(check_report.loc[check_report["severity"] == "critical", "violations"].sum())
if critical_violations == 0 and len(failed_metrics) == 0:
    conclusion = "Conditionally suitable for exploratory analysis and modeling after handling missing values and reviewing warnings."
else:
    conclusion = "Conditionally suitable only with documented restrictions; resolve critical target/identity issues before price modeling."
print("Conclusion:", conclusion)
print("Main likely limitations to discuss from the measured tables: missing fields, candidate-key duplicates, heuristic consistency rules, and unknown extraction date/source.")

### Пропонований data contract із джерелом
1. Кожен запис має містити `Suburb`, `Address`, дату події й метод продажу; комбінація передається як кандидатний ключ, а легітимні повтори документуються.
2. `Price` подається в AUD як додатне числове значення або як явний null із кодом причини; нуль не використовується як заглушка.
3. Дати передаються в ISO 8601 (`YYYY-MM-DD`), разом із датою вивантаження та ідентифікатором джерела.
4. Категорії `Type` та `Method` мають супроводжуватися узгодженим словником; зміни словника версіонуються.
5. Джерело публікує схему, одиниці вимірювання та очікувані діапазони числових полів; записи, що не проходять критичні перевірки, відхиляються або виправляються з журналом.

### Декларація використання ШІ
Для підготовки структури ноутбука та коду використано OpenAI Codex. Призначення: допомога з реалізацією профілювання, метрик і перевірок за вимогами завдання. Перевірка: код слід виконати на наданому CSV, переглянути обчислені приклади порушень і вручну підтвердити семантичні припущення перед здачею. Автор роботи має бути готовим пояснити та змінити кожну перевірку.